# Remaining Time Prediction Prototype

This Jupyter Notebook demonstrates the functionality of our trained model on new data.
In a real-life scenario this prototype could be integrated into an incident management dashboard via a REST-API.
It could receive .json data from an active ticket, apply preprocessing and predict the remaining time in real-time.

## Procedure
1. Setup and Loading of the Trained Model
2. Instantiation of an Example Case
3. Real-Time Prediction

## Setup and Loading of the Trained Model

In [ ]:
import sys
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
# append path to our code
sys.path.append('../src')
from src.feature_engineering import extract_static_case_attr, extract_aggr_dynamic_features, extract_temporal_features, encode_categorical_features
from joblib import load

print("--- Initialising prototype ---")
try:
    scaler = load('../artifacts/sscaler.pkl')
    model = load('../artifacts/reg_ols_pipeline.pkl') # This can be edited to any of our models
    expected_columns = model.feature_names_in_
    print("StandardScaler has been loaded successfully.")
    print("Model has been loaded successfully.")
except FileNotFoundError as e:
    print(f"Error occurred while loading the files: {e}")

--- Initialising prototype ---
StandardScaler has been loaded successfully.
Model has been loaded successfully.


## Instantiation of an Example Case

To simulate a real-life scenario, we instantiate an example case based on the dataset.

In [15]:
# we select a known product and a known organization from our dataset
raw_data = [
    {"case:concept:name": "0-123456789", "concept:name": "Accepted", "impact": "Medium", "product": "PROD582", "organization involved": "Org line A2", "time:timestamp": "2010-03-31 16:00:00+00:00"},
    {"case:concept:name": "0-123456789", "concept:name": "Accepted", "impact": "Medium", "product": "PROD582", "organization involved": "Org line A2", "time:timestamp": "2010-03-31 16:30:00+00:00"},
    {"case:concept:name": "0-123456789", "concept:name": "Queued",   "impact": "Medium", "product": "PROD582", "organization involved": "Org line A2", "time:timestamp": "2010-03-31 18:30:00+00:00"},
    {"case:concept:name": "0-123456789", "concept:name": "Accepted", "impact": "Medium", "product": "PROD582", "organization involved": "Org line A2", "time:timestamp": "2010-04-07 09:00:00+00:00"},
    {"case:concept:name": "0-123456789", "concept:name": "Completed", "impact": "Medium", "product": "PROD582", "organization involved": "Org line A2", "time:timestamp": "2010-04-07 09:30:00+00:00"}
]

raw_df = pd.DataFrame(raw_data)
raw_df['time:timestamp'] = pd.to_datetime(raw_df['time:timestamp'])

print("--- Raw data received ---")
print("--- Start preprocessing ---")

processed_df = extract_static_case_attr(raw_df, "case:concept:name", ["impact", "product", "organization involved"])
processed_df = extract_aggr_dynamic_features(processed_df, "case:concept:name", "time:timestamp", "concept:name")
processed_df = extract_temporal_features(processed_df, "time:timestamp")

categorical_cols = ["concept:name", "static_impact", "static_product", "static_organization involved"]
encoded_df = encode_categorical_features(processed_df, categorical_cols)
encoded_df = encoded_df.reindex(columns=expected_columns, fill_value=0)

numeric_cols = ["event_count", "elapsed_time", "activity_count"]
encoded_df[numeric_cols] = scaler.transform(encoded_df[numeric_cols])

print("Preprocessing completed successfully.")

--- Raw data received ---
--- Start preprocessing ---
Extracting static case attributes for ['impact', 'product', 'organization involved']...
Extracting aggregated dynamic features...
Extracting temporal features...
Applying one-hot encoding to ['concept:name', 'static_impact', 'static_product', 'static_organization involved']...
Preprocessing completed successfully.


## Real-Time Prediction

We iterate through each prefix of a case and predict the remaining time.

In [16]:
print("\n" + "="*68)
print("Real-Time Remaining Time Prediction")
print("="*68)

for index, row in processed_df.iterrows():
    # get index line
    current_encoded_state = encoded_df.iloc[[index]]
    # prediction
    pred = model.predict(current_encoded_state).item()
    pred_hours = max(0, pred)

    print(f"Prefix {index+1} | Status: {row['concept:name']:<10} | Remaining Time Prediction: {pred_hours:>7.2f}h")

print("="*68)


Real-Time Remaining Time Prediction
Prefix 1 | Status: Accepted   | Remaining Time Prediction: 3643.52h
Prefix 2 | Status: Accepted   | Remaining Time Prediction: 3651.68h
Prefix 3 | Status: Queued     | Remaining Time Prediction: 3725.54h
Prefix 4 | Status: Accepted   | Remaining Time Prediction: 3543.30h
Prefix 5 | Status: Completed  | Remaining Time Prediction: 3320.59h


At first glance these results seem to be blown out of proportion, but at closer inspection they're indeed accurate.
The organization we chose for the example case ("Org line A2") is known by our model for taking quite some time for their tickets.
Also, tickets can be reopened and go back to "Accepted" or "Queued" even when they have been "Completed".
Tickets only terminate upon final closure. It has also learned that a ticket being "Queued" usually means that additional time is needed, while being "Accepted" multiple times is usually associated with being "In Progress" so the time naturally reduces.
Therefore, our model provides a coherent prediction on these examples prefixes.